# CS156 · Session 9 — Metrics and Cross-Validation

This notebook reconstructs the **Generated Forum Code Workbook** for the session on evaluating classifiers beyond training accuracy.

**Main ideas:** stratified train/test splitting, logistic regression versus a nonlinear SVM on concentric circles, ROC curves and ROC-AUC, threshold tradeoffs, validation sets, and hyperparameter search.

Run the cells **top to bottom**. The circles dataset is synthetic and requires no external downloads.

## Assignment 1 planning notes

**Dataset:** Brawl Stars 3v3 match compositions. The planned target is 1 when the blue team wins and 0 when the red team wins.

**Possible failure modes:**
- too little data,
- learning a shortcut such as the user's character or a specific map instead of team-composition strength,
- ignoring player skill on specific characters,
- ignoring player synergy,
- always predicting the blue team when the user's historical win rate is above 50%.

**Possible sanity checks:**
- use only the user's character as a predictor,
- test intentionally one-sided datasets,
- test random 50/50 outcomes,
- flip the sides,
- use matches from other players,
- test on a different meta.

## 1. Generate the circles dataset and create an 80/20 train/test split

The class uses a simple binary classification dataset made of two noisy concentric circles. Class 0 is the outer circle and class 1 is the inner circle.

In [ ]:
from sklearn import datasets, model_selection
import matplotlib.pyplot as plt
import numpy as np

NUM_DATAPOINTS = 300

x_data, y_data = datasets.make_circles(
    n_samples=NUM_DATAPOINTS,
    noise=0.15,
    factor=0.6,
)

# Plot the points into a scatterplot
fig, ax = plt.subplots(figsize=(7, 7))
scatter = ax.scatter(
    x_data[:, 0],
    x_data[:, 1],
    c=y_data,
    edgecolors="grey",
    cmap="Blues",
    vmin=0,
    vmax=1,
)
ax.set_xlabel("Feature 0")
ax.set_ylabel("Feature 1")
ax.legend(
    handles=scatter.legend_elements()[0],
    labels=["outside (0)", "inside (1)"],
)
plt.show()

In [ ]:
x_train, x_test, y_train, y_test = model_selection.train_test_split(
    x_data,
    y_data,
    test_size=0.2,
    random_state=42,
    stratify=y_data,
)

### Question 1 — What kind of split is this?

This is a **stratified random split**: observations are assigned randomly, while `stratify=y_data` preserves approximately the same class proportions in train and test.

A split such as Feature 0 > 0 for training and Feature 0 < 0 for testing would likely be worse for generalization because the model would be evaluated on a region of feature space it never saw during training.

## 2. Logistic regression versus a nonlinear SVM

The circles do not have a linear decision boundary. Logistic regression therefore struggles, while an SVM with a nonlinear RBF kernel can represent a circular boundary much better.

In [ ]:
from sklearn import linear_model, metrics, svm

linear_clf = linear_model.LogisticRegression()
linear_clf.fit(x_train, y_train)

svm_clf = svm.SVC(gamma=2, C=1, probability=True)
svm_clf.fit(x_train, y_train)

print(linear_clf.score(x_test, y_test), svm_clf.score(x_test, y_test))

### Visualize the classifiers' predicted probabilities

In [ ]:
def plot_decision_surface(clf, xs, ys, ax, title):
    xs = np.linspace(xs[0], xs[1], 100)
    ys = np.linspace(ys[0], ys[1], 100)
    x_mesh, y_mesh = np.meshgrid(xs, ys)

    vis_x = np.stack((x_mesh, y_mesh), axis=2).reshape(-1, 2)
    vis_y = clf.predict_proba(vis_x)[:, 1].reshape(100, 100)

    contour = ax.contourf(
        x_mesh,
        y_mesh,
        vis_y,
        levels=np.linspace(0, 1, 33),
        cmap="Blues",
    )
    contour.set_clim(0, 1)

    ax.scatter(
        x_data[:, 0],
        x_data[:, 1],
        c=y_data,
        edgecolors="grey",
        cmap="Blues",
        vmin=0,
        vmax=1,
    )
    ax.set_title(title)
    ax.set_xlabel("Feature 0")
    ax.set_ylabel("Feature 1")

In [ ]:
from matplotlib import cm

fig, ax = plt.subplots(ncols=2, figsize=(10, 7))
fig.colorbar(
    cm.ScalarMappable(cmap="Blues"),
    ticks=np.linspace(0, 1, 11),
    ax=ax,
)

plot_decision_surface(
    linear_clf,
    (-1.5, 1.5),
    (-1.5, 1.5),
    ax[0],
    "Logistic Regression",
)
plot_decision_surface(
    svm_clf,
    (-1.5, 1.5),
    (-1.5, 1.5),
    ax[1],
    "SVM",
)
plt.show()

## 3. ROC curves and ROC-AUC

A classifier's `predict_proba` output lets us vary the decision threshold instead of fixing it at 0.5.

For each threshold, the ROC curve plots

$$\text{TPR}=\frac{TP}{TP+FN}$$

against

$$\text{FPR}=\frac{FP}{FP+TN}.$$

A curve closer to the top-left is better. ROC-AUC summarizes performance across thresholds; 0.5 is approximately random ranking and 1.0 is perfect ranking.

In [ ]:
linear_preds = linear_clf.predict_proba(x_test)[:, 1]
svm_preds = svm_clf.predict_proba(x_test)[:, 1]

linear_curve = metrics.roc_curve(y_test, linear_preds)
svm_curve = metrics.roc_curve(y_test, svm_preds)

plt.plot(
    linear_curve[0],
    linear_curve[1],
    label="Logistic ROC-AUC: %.2f"
    % metrics.roc_auc_score(y_test, linear_preds),
)
plt.plot(
    svm_curve[0],
    svm_curve[1],
    label="SVM ROC-AUC: %.2f"
    % metrics.roc_auc_score(y_test, svm_preds),
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

### Question 2 — Interpreting the ROC plot from the completed PCW

In the recorded run, the SVM had ROC-AUC **0.95** and logistic regression had ROC-AUC **0.49**, so the SVM was clearly better.

For that particular plot, the SVM curve stayed above the logistic-regression curve, so there was no operating region where logistic regression achieved a higher TPR for the same FPR. This is **not guaranteed in general**: ROC curves can cross.

At TPR = 1, the recorded plot had approximately:
- SVM: FPR ≈ 0.6,
- Logistic regression: FPR ≈ 0.8.

Because the dataset is balanced, when TPR = 1,

$$\text{precision}=\frac{1}{1+\text{FPR}}.$$

That gives roughly 61% precision for the SVM and 56% for logistic regression in the recorded run.

## 4. Why use a validation set?

The training set fits model parameters. The **validation set** is used to choose hyperparameters such as `gamma` and `C`. The **test set** should remain untouched until the model-selection process is finished.

If we repeatedly choose hyperparameters based on test-set performance, we indirectly overfit our choices to the test set. Its score then stops being an unbiased estimate of generalization performance.

## 5. Create a validation split and tune the SVM

The original 80% training partition is split again 80/20, yielding an effective **64% train / 16% validation / 20% test** split of the full dataset.

The grid search below tests every pair from six candidate `gamma` values and six candidate `C` values. The best pair is selected **only from validation ROC-AUC**. The final model is then retrained on the full original training set and evaluated once on the untouched test set.

In [ ]:
# Split the original training set again: 80% training, 20% validation
x_train_sub, x_val, y_train_sub, y_val = model_selection.train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

# Hyperparameter values to test
gamma_values = [0.1, 0.5, 1, 2, 5, 10]
C_values = [0.1, 0.5, 1, 2, 5, 10]

best_auc = -1
best_gamma = None
best_C = None

# Grid search over all gamma/C combinations
for gamma in gamma_values:
    for C in C_values:
        model = svm.SVC(
            gamma=gamma,
            C=C,
            probability=True,
        )

        model.fit(x_train_sub, y_train_sub)

        val_preds = model.predict_proba(x_val)[:, 1]
        val_auc = metrics.roc_auc_score(y_val, val_preds)

        if val_auc > best_auc:
            best_auc = val_auc
            best_gamma = gamma
            best_C = C

print("Best gamma:", best_gamma)
print("Best C:", best_C)
print("Validation ROC-AUC:", best_auc)

# Retrain using the full original training set with the best hyperparameters
best_svm = svm.SVC(
    gamma=best_gamma,
    C=best_C,
    probability=True,
)

best_svm.fit(x_train, y_train)

# Evaluate only once on the untouched test set
test_preds = best_svm.predict_proba(x_test)[:, 1]
test_auc = metrics.roc_auc_score(y_test, test_preds)

print("Test ROC-AUC:", test_auc)

### Recorded result from the completed PCW

The captured run reported:
- **Best gamma:** 5
- **Best C:** 0.1
- **Validation ROC-AUC:** 0.9913194444444444
- **Test ROC-AUC:** 0.9488888888888889

Because `make_circles` was called without a fixed `random_state`, rerunning the notebook can generate a different synthetic sample and therefore slightly different scores or selected hyperparameters.